In [1]:
import os
import json

import openai

from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue
from langsmith import Client as LangSmithClient

### Download all data from Qdrant

In [2]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [3]:
all_points = qdrant_client.scroll(
    collection_name="Amazon-items-collection-01",
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False
)

In [5]:
all_points[0][0].payload

{'title': "Couple's Bible Study Activities: 70 Engaging Activities to Connect With Your Faith and Each Other",
 'description': "Couple's Bible Study Activities: 70 Engaging Activities to Connect With Your Faith and Each Other Deepen your relationship as you strengthen your faith God's Word can give you and your partner the clarity and confidence you need to navigate life as a couple, whether you've been together for two months or 20 years. This couple's Bible study draws on the Lord's lessons to help you grow in your relationship with Him and with each other. See God's wisdom in action as you explore passages from Scripture paired with hands-on activities and conversation starters that encourage you to learn and pray together. Interactive activities—Connect with your partner through meaningful activities, like creating an at-home escape room that encourages you to work together toward a common goal. Interactive activities —Connect with your partner through meaningful activities, like c

In [6]:
all_context = [{"id": data.payload["parent_asin"], "text": data.payload["description"]} for data in all_points[0]]

In [7]:
all_context

[{'id': '1638787158',
  'text': "Couple's Bible Study Activities: 70 Engaging Activities to Connect With Your Faith and Each Other Deepen your relationship as you strengthen your faith God's Word can give you and your partner the clarity and confidence you need to navigate life as a couple, whether you've been together for two months or 20 years. This couple's Bible study draws on the Lord's lessons to help you grow in your relationship with Him and with each other. See God's wisdom in action as you explore passages from Scripture paired with hands-on activities and conversation starters that encourage you to learn and pray together. Interactive activities—Connect with your partner through meaningful activities, like creating an at-home escape room that encourages you to work together toward a common goal. Interactive activities —Connect with your partner through meaningful activities, like creating an at-home escape room that encourages you to work together toward a common goal. Theme

### Render a prompt to generate synthetic Eval reference dataset

In [14]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "Sugested question.",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "ID of the chunk that could be used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in teh context,",
            },
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the questions could be answered with the chunks.",
            },
        },
    },
}

SYSTEM_PROMPT = f"""
I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have avaialable.
I will provide all of the available products to you with IDs of each chunk.
Come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitatea potentjial real user of this RAG system - a customerof the e-shop.
As an outputI need you to provide me with listof questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answerto the question given teh context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multople chunks in the questions.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that cant' be answered with the available chunks.
Don't use word chunks in suggested questions, refer to the chunks as products.
Don't add markdown formatting strings in the answer.

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON SCHEMA>

I need to be able to parse the json output.
"""

USER_PROMPT = f"""
Here is the list of chunks, each lit element is a dictionary with id and text:
{json.dumps(all_context, indent=2)}
"""

In [15]:
response = openai.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT}
    ],
    reasoning_effort="none"
)

print(response.choices[0].message.content)

[
  {
    "question": "Which book is a good choice if I want a romantic story set in Regency England?",
    "chunk_ids": ["1728261961"],
    "answer_example": "The Rogue Steals a Bride is set in the glittering halls and glamorous balls of Regency England, so it would be a good choice for a romantic story in that setting.",
    "reasoning": "This can be answered from a single product because the description explicitly states the setting and genre."
  },
  {
    "question": "Do you have a beginner-friendly guide for learning bonsai care?",
    "chunk_ids": ["1915577012"],
    "answer_example": "Yes, Bonsai for Beginners is an easy reference care guide with step-by-step instructions, beginner-friendly species, maintenance techniques, and pruning and repotting guides.",
    "reasoning": "The product description directly says it is for beginners and lists care instructions and tools."
  },
  {
    "question": "Is there a cookbook that focuses specifically on duck recipes?",
    "chunk_ids":

In [16]:
json_output = json.loads(response.choices[0].message.content)

In [17]:
len(json_output)

33

In [20]:
single_chunk_counter = 0
multiple_chunk_counter = 0
zero_chunk_counter = 0

for question in json_output:
    if len(question["chunk_ids"]) == 1:
        single_chunk_counter += 1
    elif len(question["chunk_ids"]) > 1:
        multiple_chunk_counter += 1
    else:
        zero_chunk_counter += 1

print(f"""
Questions with:
 - zero chunks: {zero_chunk_counter}
 - single chunk: {single_chunk_counter}
 - multiple chunks: {multiple_chunk_counter}
""")


Questions with:
 - zero chunks: 4
 - single chunk: 18
 - multiple chunks: 11



In [ ]:
point = qdrant_client.scroll(
    collection_name="Amazon-items-collection-01",
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key="parent_asin",
                match=MatchValue(value="1915577012")
            )
        ]
    ),
    limit=100,
    with_payload=True,
    with_vectors=False
)[0]

In [26]:
point[0].payload

{'title': 'Bonsai for Beginners: An easy reference care guide book with step-by-step instructions and all the best tools and techniques to get you to success with your first Bonsai Tree',
 'description': 'Bonsai for Beginners: An easy reference care guide book with step-by-step instructions and all the best tools and techniques to get you to success with your first Bonsai Tree Embark on a captivating journey into the timeless art of bonsai with "Bonsai for Beginners" - your ultimate companion to nurturing your first bonsai tree with grace and ease. Responding to our valued readers\' feedback, we\'ve significantly enhanced this edition. It now boasts an expanded gallery of images, enriched tips, deeper insights into essential techniques, and much more, all designed to ensure your bonsai flourishes. Whether you\'re a green-thumbed veteran or a gardening novice, this book is your roadmap to success. It\'s packed with stunning illustrations, vivid photographs, and straightforward instructi

In [28]:
def get_description(parent_asin: str) -> str:
    point = qdrant_client.scroll(
        collection_name="Amazon-items-collection-01",
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="parent_asin",
                    match=MatchValue(value=parent_asin)
                )
            ]
        ),
        limit=100,
        with_payload=True,
        with_vectors=False
    )[0][0]

    return point.payload["description"]

In [30]:
get_description("1915577012")

'Bonsai for Beginners: An easy reference care guide book with step-by-step instructions and all the best tools and techniques to get you to success with your first Bonsai Tree Embark on a captivating journey into the timeless art of bonsai with "Bonsai for Beginners" - your ultimate companion to nurturing your first bonsai tree with grace and ease. Responding to our valued readers\' feedback, we\'ve significantly enhanced this edition. It now boasts an expanded gallery of images, enriched tips, deeper insights into essential techniques, and much more, all designed to ensure your bonsai flourishes. Whether you\'re a green-thumbed veteran or a gardening novice, this book is your roadmap to success. It\'s packed with stunning illustrations, vivid photographs, and straightforward instructions, empowering you to master this exquisite art form swiftly and enjoyably. Within the pages of "Bonsai for Beginners," you\'ll uncover: 3 beginner-friendly bonsai tree species to kickstart your journey 

### Create Eval dataset in LangSmith

In [31]:
ls_client = LangSmithClient(api_key=os.getenv("LANGSMITH_API_KEY"))

In [32]:
dataset_name = "rag-evaluation-dataset"
dataset = ls_client.create_dataset(
    dataset_name=dataset_name,
    description="RAG evaluation dataset"
)

In [33]:
for item in json_output:
    ls_client.create_example(
        dataset_id=dataset.id,
        inputs={
            "question": item["question"]
        },
        outputs={
            "ground_truth": item["answer_example"],
            "reference_context_ids": item["chunk_ids"],
            "reference_descriptions": [get_description(id) for id in item["chunk_ids"]]
        }
    )